# LFM Instance Segmentation Example Workflow
This notebook trains a Graha/Lunar-FM Mask R-CNN instance segmentation model for crater detection. It loads a split crater instance dataset, builds the Graha object-detection datamodule and TerraTorch task, runs fine-tuning, writes checkpoints, and creates validation prediction plots.

## Purpose of this notebook
Use this notebook as the active interactive Graha instance-segmentation training workflow. The values in the **User Configuration** section mirror the most commonly changed command-line options; lower-level options stay on the centralized experiment-config defaults unless they are explicitly promoted into that section.

**Note**: dataset-specific image/label matching, band selection, normalization modality, and Graha input modality are controlled by `DATA_DICT` in the **User Configuration** section. See the repository README for dataset-specific examples.


## Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

import sys

from functools import partialmethod
from glob import glob
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import torch
from lightning.pytorch import seed_everything
from tqdm import tqdm

tqdm.__init__ = partialmethod(tqdm.__init__, disable=False)

In [ ]:
repo_root = Path.cwd().parent
NOTEBOOK_DIR = repo_root / "notebooks"

if not (repo_root / "lfm").exists():
  raise FileNotFoundError(
      "Cannot find lfm/ directory. Run this notebook from "
      "lfm/notebooks/full_model or update repo_root."
  )

sys.path.insert(0, str(repo_root))

from lfm.all_models.all_tasks.utils import (
  create_timestamped_output_dir,
  plot_instance_cache_predictions,
  save_graha_instance_prediction_cache,
)
from lfm.all_models.inst_seg import build_graha_notebook_configs
from lfm.full_model.inst_seg import instance_graha_components

print("Successfully imported LFM modules")

## User Configuration

These are the values a notebook user is expected to edit for a normal Graha instance-segmentation run.

### Paths
`BASE_OUTPUT_DIR`: parent directory for timestamped notebook outputs. Checkpoints, config files, prediction caches, and plots are written under a new timestamped subdirectory.

`DATA_DIR`: split dataset root. It should contain `train/`, `val/`, and `test/` folders, each with `chips/` and `labels/` subfolders.

`PRETRAIN_DIR`: Graha/Lunar-FM pretraining directory. It must contain `checkpoints/checkpoint_weights_final.pt`, `full_config.yaml`, and `modality_info.yaml`.

`LIGHTNING_CHECKPOINT`: optional Graha Lightning checkpoint to resume from. Leave as `None` for a fresh fine-tune.

### Data Selection
To switch datasets, update `DATA_DIR` and replace `DATA_DICT` with the matching dictionary from the README, under **"Dataset Specifications"**.

`DATA_DICT`: dataset-level dictionary that controls the data-specific parts of training. It defines the dataset modality, chip and label glob patterns, selected chip bands, normalization modality when needed, and Graha input modality. This replaces the older pattern of separately editing `DATASET_MODALITY`, `BAND_FILTER`, `IMAGE_GLOB`, `LABEL_GLOB`, `IMAGE_SUFFIX`, `LABEL_SUFFIX`, `NORMALIZATION_MODALITY`, and Graha modality flags.

`DATA_DICT["band_filters"]`: modality-local band selection. For WAC, `"vis": [0, 1, 2, 3, 4]` and `"uv": [0, 1]` selects all 7 stored WAC channels. For NAC PHO or DTM, use `[0]` because each modality is stored as a single band.

`MAX_TRAIN_SAMPLES`, `MAX_VAL_SAMPLES`, `MAX_TEST_SAMPLES`: optional split caps for quick experiments. Set any of these to `None` to use the full split.

### Training
`BATCH_SIZE`: Graha training batch size.

`NUM_WORKERS`: dataloader worker count.

`MAX_EPOCHS`: number of fine-tuning epochs.

`GRAHA_BACKBONE_LR`, `GRAHA_HEAD_LR`, `GRAHA_LAYER_DECAY`, `GRAHA_WEIGHT_DECAY`, `GRAHA_WARMUP_STEPS`: Graha optimizer schedule parameters.

### Defaults Kept In Code
The notebook leaves these centralized defaults unchanged unless you intentionally add overrides to the config cell. File suffixes are inferred automatically from common names such as `_input_nac_chip`, `_input_wac_chip`, `_label`, `_mask`, and `_img`; add explicit `image_suffix` or `label_suffix` only for unusual datasets. Normalization source defaults to `"pretrain"`. Dataset-specific defaults should normally stay inside `DATA_DICT`: `TARGET_SIZE=256`, `GRAHA_STATS_BATCH_SIZE=16`, `GRAHA_VIS_UV_MERGE_METHOD="mean"`, `GRAHA_ANCHOR_SIZES=[[8], [16], [32], [64]]`, `GRAHA_ANCHOR_ASPECT_RATIOS=[0.5, 1.0, 2.0]`, `GRAHA_SCORE_THRESHOLD=0.5`, `PLOT_EVERY_N_EPOCHS=1`, `PLOT_N_SAMPLES=5`, `PREDICTION_SPLIT="val"`, `PREDICTION_N_SAMPLES=5`, `PREDICTION_SCORE_THRESHOLD=0.5`, `MASK_SHIFT=(0, 0)`, `IGNORE_NODATA_IN_LOSS=False`, `NODATA_IGNORE_INDEX=-1`, `SEED=42`.

In [ ]:
BASE_OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "instance_seg_finetuning"
DATA_DIR = "/explore/nobackup/projects/lfm/model_inputs/300_300_inputs/full_model_inst_seg_v2"
PRETRAIN_DIR = "/explore/nobackup/projects/lfm/ibm_model_pretrain_dir"
LIGHTNING_CHECKPOINT = None

DATA_DICT = {
    "dataset_name": "nac_craters_pho_dtm",
    "dataset_modality": "nac_dtm",
    "image_glob": "*.tif",
    "label_glob": "*_label.npz",
    "semantic_label_source": "instance",
    "band_filters": {
        "pho": [0],
        "dtm": [0],
    },
    "normalization_modality": "nac",
    "graha_input_modalities": ["nac", "dtm"],
}

MAX_TRAIN_SAMPLES = 500
MAX_VAL_SAMPLES = 500
MAX_TEST_SAMPLES = 500

BATCH_SIZE = 8
NUM_WORKERS = 10
MAX_EPOCHS = 1

GRAHA_BACKBONE_LR = 5.0e-5
GRAHA_HEAD_LR = 2.0e-4
GRAHA_LAYER_DECAY = 0.75
GRAHA_WEIGHT_DECAY = 0.05
GRAHA_WARMUP_STEPS = 500

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

The configuration cell above mirrors the active Graha instance-segmentation training settings. Values not listed there use the centralized defaults documented in the previous markdown cell.


In [ ]:
OUTPUT_DIR = create_timestamped_output_dir(BASE_OUTPUT_DIR)

notebook_configs = build_graha_notebook_configs(
    output_dir=OUTPUT_DIR,
    data_root=DATA_DIR,
    base_output_dir=OUTPUT_DIR,
    graha_pretrain_dir=PRETRAIN_DIR,
    graha_lightning_checkpoint=LIGHTNING_CHECKPOINT,
    data_dict=DATA_DICT,
    max_epochs=MAX_EPOCHS,
    graha_batch_size=BATCH_SIZE,
    graha_num_workers=NUM_WORKERS,
    max_train_samples=MAX_TRAIN_SAMPLES,
    max_val_samples=MAX_VAL_SAMPLES,
    max_test_samples=MAX_TEST_SAMPLES,
    graha_backbone_lr=GRAHA_BACKBONE_LR,
    graha_head_lr=GRAHA_HEAD_LR,
    graha_layer_decay=GRAHA_LAYER_DECAY,
    graha_weight_decay=GRAHA_WEIGHT_DECAY,
    graha_warmup_steps=GRAHA_WARMUP_STEPS,
)

config = notebook_configs.experiment_config
graha_config = notebook_configs.graha_config
deps = notebook_configs.dependencies

seed_everything(config.seed)
instance_graha_components.save_config(graha_config, OUTPUT_DIR)

print("Config created successfully")
print(f"Data dir: {config.data_root}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Graha modality mode: {config.graha_input_modality_mode}")
print(f"Normalization modality: {config.normalization_modality}")

## Output Directory
The timestamped output directory was created while building the config. It contains the saved config, checkpoints, prediction caches, and plots for this run.


In [ ]:
print(f"Notebook output directory: {OUTPUT_DIR}")

## Create datamodule
1. Load pretraining stats from .yaml file
2. Create datamodule using pretraining stats and other config options

In [ ]:
# STEP 1: pretraining stats
print("\nSTEP 1: Loading pretraining stats...")
print("="*60)

datamodule_cls = deps["GrahaObjectDetectionInstanceDataModule"]
means, stds = instance_graha_components.get_normalization_stats(
    graha_config,
    datamodule_cls,
)

print("Done.")

In [ ]:
print("\nSTEP 2: Creating datamodule and inspecting one training batch...")
print("=" * 60)

graha_datamodule = instance_graha_components.create_datamodule(
    graha_config,
    datamodule_cls,
    means,
    stds,
)
graha_sample_batch = instance_graha_components.inspect_batch(graha_datamodule)

print("Done.")

## Create Terratorch Task Object, Model

In [ ]:
task_cls = instance_graha_components.make_downstream_object_detection_task_class(
    deps["LunarObjectDetectionTask"]
)

graha_task = instance_graha_components.create_task(
    graha_config,
    task_cls,
    graha_sample_batch,
)
instance_graha_components.run_loss_smoke(graha_task, graha_sample_batch)

## Run Training

In [ ]:
trainer = instance_graha_components.create_trainer(graha_config, OUTPUT_DIR)
print(trainer)

In [ ]:
print("\n" + "=" * 60)
print("Starting training.")
print("=" * 60)

ckpt_path = (
    str(graha_config.lightning_checkpoint)
    if graha_config.lightning_checkpoint is not None
    else None
)
trainer.fit(
    graha_task,
    datamodule=graha_datamodule,
    ckpt_path=ckpt_path,
)

print("Finished training.")

## Create And Display Validation Visualizations
Using the saved checkpoint, this section inferences/predicts on the reserved validation dataset and displays a visualization for review


In [ ]:
prediction_cache = save_graha_instance_prediction_cache(
    task=graha_task,
    datamodule=graha_datamodule,
    output_dir=OUTPUT_DIR,
    model_name="graha",
    split=config.prediction_split,
    n_samples=config.prediction_n_samples,
    score_threshold=config.prediction_score_threshold,
)

prediction_plot = plot_instance_cache_predictions(
    prediction_cache,
    OUTPUT_DIR / "plots" / "single_model" / "graha_model",
    model_name="graha",
    n_samples=config.prediction_n_samples,
    filename=f"{config.prediction_split}_instance_predictions.png",
)
print(f"Saved prediction plot: {prediction_plot}")

In [ ]:
img = mpimg.imread(prediction_plot)
plt.figure(figsize=(16, 14))
plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
del graha_task, graha_datamodule, trainer
if torch.cuda.is_available():
    torch.cuda.empty_cache()